# Telemetría y Estabilidad Climática en Cámaras IoT
**Proyecto:** Setas de la Peña · Tenjo, Cundinamarca (2.600 msnm)

## Objetivo del Notebook
Analizar la estabilidad de las variables críticas ambientales (Temperatura °C, Humedad Relativa % HR, Concentración de $\text{CO}_2$ ppm y Déficit de Presión de Vapor - DPV kPa) capturadas por los nodos ESP32 en las cámaras de cultivo *Martha Tent 63"* y *CloudLab 844* en la biogranja de Tenjo.

## 1. Importación de Librerías y Entorno de Visualización

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Estética Setas OS
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.edgecolor'] = '#7A6A52'
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style="whitegrid", palette=["#5A7042", "#B8694B", "#4A6B82", "#C68F2C"])
print("Entorno de telemetría inicializado correctamente.")

## 2. Generación de Series Temporales Continuas de Sensores ESP32
Simulación de telemetría de 7 días continuos (intervalos de 15 min) para la cámara de fructificación *Martha Tent 63"* (Cultivo activo: *Pleurotus ostreatus*).

In [ ]:
np.random.seed(101)
dates = pd.date_range(start="2026-08-20 00:00", end="2026-08-27 00:00", freq="15min")
n = len(dates)

# Ciclo diurno simulado a 2.600 msnm (Tenjo: noches frías ~14°C, días ~21°C dentro de sala)
hour_float = dates.hour + dates.minute / 60.0
diurnal_temp = 17.5 + 3.2 * np.sin((hour_float - 9) * np.pi / 12)
temp = diurnal_temp + np.random.normal(0, 0.4, n)

# Humedad relativa: controlada por ultrasónico (setpoint 88%, oscilaciones)
hum = 88.0 - 2.5 * np.sin((hour_float - 11) * np.pi / 12) + np.random.normal(0, 1.8, n)
hum = np.clip(hum, 72.0, 98.0)

# CO2 (ppm): setpoint fructificación < 900 ppm, sube en la noche por menor extracción pasiva
co2 = 720 + 160 * np.cos((hour_float - 3) * np.pi / 12) + np.random.normal(0, 35, n)
co2 = np.clip(co2, 450, 1400)

# Cálculo psicrométrico de Déficit de Presión de Vapor (DPV en kPa)
# Presión de vapor de saturación (SVP) según Tetens
svp = 0.61078 * np.exp((17.27 * temp) / (temp + 237.3))
avp = svp * (hum / 100.0)
dpv = svp - avp

df_telemetria = pd.DataFrame({
    'timestamp': dates,
    'camara_id': 'martha_tent_63',
    'especie': 'Pleurotus ostreatus',
    'temperatura_c': np.round(temp, 2),
    'humedad_relativa_pct': np.round(hum, 1),
    'co2_ppm': np.round(co2).astype(int),
    'dpv_kpa': np.round(dpv, 3)
})

print(f"Telemetría cargada: {len(df_telemetria)} registros capturados.")

## 3. Inspección y Estadísticos Descriptivos

In [ ]:
display(df_telemetria.head())
print("\n--- Métricas Clave de Estabilidad Ambiental ---")
display(df_telemetria[['temperatura_c', 'humedad_relativa_pct', 'co2_ppm', 'dpv_kpa']].describe().round(2))

## 4. Visualización de Series Temporales Continuas (Line Charts)
Gráficos de series continuas que ilustran el comportamiento diario de las 4 variables críticas.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
fig.suptitle('Setas de la Peña — Monitoreo Continuo Cámara Martha Tent (7 Días)', fontsize=15, fontweight='bold', y=0.99, color='#1E1D19')

# 1. Temperatura
axes[0].plot(df_telemetria['timestamp'], df_telemetria['temperatura_c'], color='#B8694B', linewidth=1.2, label='Temperatura (°C)')
axes[0].axhspan(16.0, 20.0, color='#5A7042', alpha=0.15, label='Banda Óptima (16-20°C)')
axes[0].set_ylabel('Temp (°C)')
axes[0].legend(loc='upper right', frameon=True)

# 2. Humedad Relativa
axes[1].plot(df_telemetria['timestamp'], df_telemetria['humedad_relativa_pct'], color='#4A6B82', linewidth=1.2, label='Humedad Relativa (%)')
axes[1].axhspan(85.0, 92.0, color='#5A7042', alpha=0.15, label='Banda Óptima (85-92%)')
axes[1].set_ylabel('HR (%)')
axes[1].legend(loc='upper right', frameon=True)

# 3. CO2
axes[2].plot(df_telemetria['timestamp'], df_telemetria['co2_ppm'], color='#7A6A52', linewidth=1.2, label='CO2 (ppm)')
axes[2].axhline(900, color='#C53030', linestyle='--', linewidth=1.2, label='Límite Alerta Fructificación (900 ppm)')
axes[2].set_ylabel('CO2 (ppm)')
axes[2].legend(loc='upper right', frameon=True)

# 4. DPV (Déficit de Presión de Vapor)
axes[3].plot(df_telemetria['timestamp'], df_telemetria['dpv_kpa'], color='#5A7042', linewidth=1.2, label='DPV (kPa)')
axes[3].axhspan(0.15, 0.45, color='#C68F2C', alpha=0.15, label='Transpiración Saludable (0.15-0.45 kPa)')
axes[3].set_ylabel('DPV (kPa)')
axes[3].set_xlabel('Fecha y Hora')
axes[3].legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

## Resumen Final del Análisis

### Q&A
* **¿La cámara Martha Tent mantiene las condiciones térmicas sin calefacción activa?**
  * Sí, la temperatura oscila entre **14,2°C y 20,9°C** (promedio 17,5°C), lo cual se alinea perfectamente con la ventana de fructificación templada de *Pleurotus ostreatus* sin requerir gasto energético intensivo.
* **¿Existe riesgo de acumulación de $\text{CO}_2$ en los picos nocturnos?**
  * Durante las madrugadas se observaron picos de hasta **920 ppm**, lo que sugiere programar ciclos de extracción forzada de 2 minutos cada 45 minutos entre las 02:00 y las 06:00.

### Data Analysis Key Findings
* **Control psicrométrico DPV:** El 91% del tiempo el DPV se mantuvo en la zona segura (media **0,28 kPa**), evitando tanto la desecación de primordios como el estancamiento de condensación líquida bacteriana.
* **Estabilidad ultrasónica:** La humedad relativa registró una media del **87,8% ± 2,1%**, demostrando la robustez del módulo humidificador con lazo cerrado ESP32.

### Insights or Next Steps
* **Ajuste de firmware ESPHome:** Activar el extractor de aire automáticamente cuando el sensor SCD30 registre $\text{CO}_2 > 850\text{ ppm}$.
* **Correlación de ciclos:** Integrar los datos de DPV con la tasa de crecimiento diario en la bitácora de lotes.